# Connecting to DB

In [ ]:
# import os
# from dotenv import load_dotenv
# from sqlalchemy import create_engine, text

# # Load key/value settings from .env in the workspace root.
# load_dotenv()

# host = os.getenv("AIVEN_PG_HOST")
# port = os.getenv("AIVEN_PG_PORT", "22872")
# user = os.getenv("AIVEN_PG_USER")
# password = os.getenv("AIVEN_PG_PASSWORD")
# database = os.getenv("AIVEN_PG_DATABASE")
# sslmode = os.getenv("AIVEN_PG_SSLMODE", "require")

# uri = (
#     f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
#     f"?sslmode={sslmode}"
# )

# engine = create_engine(uri, pool_pre_ping=True)

# with engine.connect() as conn:
#     row = conn.execute(text("SELECT current_database(), current_user")).fetchone()
#     print(f"Connected to DB: {row[0]} as user: {row[1]}")

Connected to DB: sponsor_db as user: avnadmin


In [2]:
import pandas as pd
from tqdm import tqdm
from google import genai
from google.genai import types

# Read Data

### Read and Complete df_companies

In [7]:
df_companies = pd.read_csv("data_source/companies_data.csv")
df_companies.head()

,Company ID,Stock Code,Company Name,Industry,Propensity Score,Address,Website,LinkedIn,Annual Revenue (HKD),Marketing Budget Est.,Social Media Followers,Engagement Rate,Market Cap Bn,Company Size,Contact Person,Email,Phone
0,101,0293.HK,Cathay Pacific,Aviation,88.0,"Cathay City, 8 Scenic Road, HKIA, Lantau",www.cathaypacific.com,linkedin.com/company/cathay-pacific,$85B,$500M,2.5M,4.20%,79.12,23000.0,Head of Corporate Affairs,press@cathaypacific.com,85227471888
1,102,0005.HK,HSBC Hong Kong,Banking & Finance,95.0,"1 Queen's Road Central, Hong Kong",www.hsbc.com.hk,linkedin.com/company/hsbc,$400B,$1.2B,3.1M,3.50%,2306.60,214000.0,Group Corporate Communications,mediarelations@hsbc.com,85228221111
2,103,1299.HK,AIA Group,Insurance,82.0,"1 Connaught Road Central, Hong Kong",www.aia.com.hk,linkedin.com/company/aia,$250B,$800M,1.8M,2.80%,882.96,25938.0,Group Corporate Communications (Media),mediarelations@aia.com,85222328888.0
3,104,0008.HK,PCCW,Telecommunications,76.0,"PCCW Tower, Taikoo Place, Quarry Bay",www.pccw.com,linkedin.com/company/pccw,$38B,$200M,800K,1.90%,46.18,20000.0,Group Corporate Communications,corpcomms@pccw.com,85228882888
4,105,1929.HK,Chow Tai Fook,Retail & Luxury,85.0,"38th Floor, New World Tower, Central",www.chowtaifook.com,linkedin.com/company/chow-tai-fook,$70B,$350M,1.2M,5.10%,119.17,38000.0,Corporate Communications Department,ir@chowtaifook.com,85225268632


In [10]:
class Gemini_Company_Info_Extractor:
    def __init__(self):
        self.client = genai.Client()
        self.system_instruction = """
            You are expert in searching information from listed company. According the below example, please extract the key information about the company, with exact JSON format. The example:
            'Stock Code': '1299.HK',
            'Company Name': 'AIA Group',
            'Industry': 'Insurance',
            'Propensity Score': 82.0,
            'Address': '1 Connaught Road Central, Hong Kong',
            'Website': 'www.aia.com.hk',
            'LinkedIn': 'linkedin.com/company/aia',
            'Annual Revenue (HKD)': '$250B',
            'Marketing Budget Est.': '$800M',
            'Social Media Followers': '1.8M',
            'Engagement Rate': '2.80%',
            'Market Cap Bn': 882.96,
            'Company Size': 25938.0,
            'Contact Person': 'Group Corporate Communications (Media)',
            'Email': 'mediarelations@aia.com',
            'Phone': '85222328888'
        """
    
    def get_contents(self, stock_code, company_name):
        contents = f"""
            You are helping finding Sponsorships for Sport Events in Hong Kong Jockey Club.
            Considering 'Stock Code': '{stock_code}', 'Company Name': '{company_name}'
            Find out the following information about the company:
            'Industry' - the industry the company belongs to, such as 'Finance', 'Technology', 'Retail', etc.,
            'Propensity Score' - which indicates the company's likelihood to sponsor sport events, on a scale of 0 to 100,
            'Address',
            'Website',
            'LinkedIn',
            'Annual Revenue (HKD)',
            'Marketing Budget Est.',
            'Social Media Followers',
            'Engagement Rate',
            'Market Cap Bn',
            'Company Size',
            'Contact Person' - the name of the person in charge of sponsorship or marketing,
            'Email',
            'Phone'.

            Please output the information in exact JSON format as below, and only output the JSON without any explanation or extra words.
        """
        return contents

    def extract_info(self, stock_code, company_name):
        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=self.get_contents(stock_code, company_name),
            config=types.GenerateContentConfig(system_instruction=self.system_instruction)
        )
        return response.text




In [9]:
# transform as json
import json
def transform_to_json(text):
    # extract the text between ```json and ```
    if text.startswith("```json") and text.endswith("```"):
        text = text[len("```json\n"):-len("\n```")]
    try:
        data = json.loads(text)
        return data
    except json.JSONDecodeError as e:
        print("Failed to decode JSON:", e)
        return None

In [ ]:
# For each company, we can call the extract_info function to get the information in JSON format.
extractor = Gemini_Company_Info_Extractor()
for _, row in tqdm(df_companies.iterrows()):
    # if the row is completed - pass
    if all(pd.notna(row[col]) for col in df_companies.columns):
        continue
    stock_code = row['Stock Code']  # Assuming 'Stock Code' is the first column
    company_name = row['Company Name']  # Assuming 'Company Name' is the second column
    company_info_text = extractor.extract_info(stock_code, company_name)
    company_info_json = transform_to_json(company_info_text)

    # if the cell in the row is empty, we can fill it with the extracted information
    for col in df_companies.columns:
        if pd.isna(row[col]):  # Assuming the target column for extracted info is the third column
            df_companies.at[row.name, col] = company_info_json.get(col, "")  # Replace 'Extracted Info' with the actual column name
            print(f"Filled {col} for {company_name} with extracted info. {company_info_json.get(col, '')}")


Filled Industry for CK Hutchison Holdings with extracted info. Diversified Holdings
Filled Propensity Score for CK Hutchison Holdings with extracted info. 75.0
Filled Address for CK Hutchison Holdings with extracted info. 22nd Floor, Hutchison House, 10 Harcourt Road, Central, Hong Kong
Filled Website for CK Hutchison Holdings with extracted info. www.ckh.com.hk
Filled LinkedIn for CK Hutchison Holdings with extracted info. linkedin.com/company/ck-hutchison-holdings
Filled Annual Revenue (HKD) for CK Hutchison Holdings with extracted info. $447.25B
Filled Marketing Budget Est. for CK Hutchison Holdings with extracted info. $1.5B
Filled Social Media Followers for CK Hutchison Holdings with extracted info. 100K
Filled Engagement Rate for CK Hutchison Holdings with extracted info. 1.50%
Filled Company Size for CK Hutchison Holdings with extracted info. 300000.0
Filled Contact Person for CK Hutchison Holdings with extracted info. Corporate Communications Department
Filled Email for CK Hutc

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [25]:
df_companies.to_csv("data_source/companies_data_filled.csv", index=False)

### Read df_staff

In [22]:
df_staff = pd.read_csv("data_source/staff_data.csv")

In [23]:
df_staff

,Company ID,Contact Name,Role,Connection Strength,Previous HKJC Interaction,Link of HKJC Interaction evidance,Linkedin
0,101,Ronald Lam,CEO,Warm,Yes - 2022 Gala,NaN,NaN
1,101,Edward Bell,"GM Brand, Insights & Marketing",Hot,Yes - Regular Box Owner,https://www.sportsmatters.asia/,https://www.linkedin.com/in/ecbell30
2,102,Luanne Lim,"CEO, Hong Kong",Cold,No,NaN,NaN
3,102,Suresh Balaji,CMO,Warm,Yes - 2023 Charity Event,NaN,NaN
4,103,Jacky Chan,Regional Chief Executive,Warm,No,NaN,NaN
5,104,Susanna Hui,Acting Group MD,Cold,No,NaN,NaN
6,105,Conroy Cheng,Vice-Chairman,Hot,Yes - Horse Owner,NaN,NaN
7,105,Peter Suen,Executive Director,Warm,Yes - Member,NaN,NaN


In [24]:
class Gemini_Company_Staff_Extractor:
    def __init__(self):
        self.client = genai.Client()
        self.system_instruction = """
            You are expert in searching information of key staffs from listed company accurately. According the below example, please extract the key information about key staffs of the company, with exact JSON format. The example:
            'Stock Code': '0293.HK',
            'Company Name': 'Cathay Pacific',
            'Contact Name': 'Edward Bell',
            'Role': 'GM Brand, Insights & Marketing',
            'Connection Strength': 'Hot',
            'Previous HKJC Interaction': 'Yes - Regular Box Owner',
            'Link of HKJC Interaction evidance': 'https://www.sportsmatters.asia/',
            'Linkedin': 'https://www.linkedin.com/in/ecbell30'
        """
    
    def get_contents(self, stock_code, company_name):
        contents = f"""
            You are helping finding key staffs who can succeed Sponsorships for Sport Events in Hong Kong Jockey Club.
            Considering 'Stock Code': '{stock_code}', 'Company Name': '{company_name}'
            Find out the following information about those key staffs of the company:
            'Industry' - the industry the company belongs to, such as 'Finance', 'Technology', 'Retail', etc.,
            'Contact Name' - the name of the key staff member,
            'Role' - their role in the company,
            'Connection Strength' - how strong their connection to the HKJC is. Either 'Hot', 'Warm' or 'Cold',
            'Previous HKJC Interaction' - whether they have interacted with HKJC before, if yes what interaction is, with not more than 20 words,
            'Link of HKJC Interaction evidance' - a link to evidence of their interaction with HKJC. If no interaction, put as 'Nil'. Please verify the link is valid and can be accessed, and the content in the link can clearly show the interaction between the key staff and HKJC. If there are multiple evidences, please provide the most recent one,
            'Linkedin' - their LinkedIn profile link. If no linkedin profile, put as 'Nil'. Please verify the link is valid and can be accessed, and the content in the link can clearly show their current role in the company. If there are multiple evidences, please provide the most recent one.

            Maybe there are more than one key staff in the company, please extract information of up to 3 key staffs.
            Please output the information in exact JSON format as below, and only output the JSON without any explanation or extra words.
        """
        return contents

    def extract_info(self, stock_code, company_name):
        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=self.get_contents(stock_code, company_name),
            config=types.GenerateContentConfig(system_instruction=self.system_instruction)
        )
        return response.text


In [30]:
staff_extractor = Gemini_Company_Staff_Extractor()

# For each company,
for _, row_company in tqdm(df_companies[:50].iterrows()):
    # if the row of company is not completed - pass
    if not all(pd.notna(row_company[col]) for col in df_companies.columns):
        continue

    company_id = row_company['Company ID']  # Assuming 'Company ID' is the first column
    stock_code = row_company['Stock Code']
    company_name = row_company['Company Name']

    # check whether the rows of df_staff with the same stock code are completed.
    df_staff_company = df_staff[df_staff['Company ID'] == company_id]
    if len(df_staff_company) < 3 or not all(df_staff_company.apply(lambda x: all(pd.notna(x[col]) for col in df_staff.columns), axis=1)):
        staff_info_text = staff_extractor.extract_info(stock_code, company_name)
        staff_info_json = transform_to_json(staff_info_text)

        # fill the information of key staffs into df_staff
        rows_staff = []
        for i in range(min(3, len(staff_info_json))):  # up to 3 key staffs
            staff_info = staff_info_json[i]
            new_row = {
                'Company ID': company_id,
                'Contact Name': staff_info.get('Contact Name', ''),
                'Role': staff_info.get('Role', ''),
                'Connection Strength': staff_info.get('Connection Strength', ''),
                'Previous HKJC Interaction': staff_info.get('Previous HKJC Interaction', ''),
                'Link of HKJC Interaction evidance': staff_info.get('Link of HKJC Interaction evidance', ''),
                'Linkedin': staff_info.get('Linkedin', '')
            }
            rows_staff.append(new_row)
            print(f"Filled staff info for {company_name}: {new_row['Contact Name']} - {new_row['Role']}")

        df_rows_staff = pd.DataFrame(rows_staff)
        df_staff = pd.concat([df_staff[df_staff['Company ID'] != company_id], df_rows_staff], ignore_index=True)

11it [00:33,  3.01s/it]

Filled staff info for Jardine Matheson: Ben Keswick - Executive Chairman
Filled staff info for Jardine Matheson: John Witt - Group Managing Director
Filled staff info for Jardine Matheson: Christian Sardelis - Group Corporate Affairs Director


12it [00:46,  4.24s/it]

Filled staff info for Sun Hung Kai Properties: Raymond Kwok Ping-luen - Chairman and Managing Director
Filled staff info for Sun Hung Kai Properties: Adam Kwok Kai-fai - Executive Director
Filled staff info for Sun Hung Kai Properties: Christopher Kwok Kai-wang - Executive Director


13it [00:57,  5.14s/it]

...

Filled staff info for China Merchants Port: Wang Jianping - Executive Director and Chief Executive Officer
Filled staff info for China Merchants Port: Deng Renjie - Chairman of the Board
Filled staff info for China Merchants Port: Lee Kwok Po, Peter - Independent Non-executive Director


49it [11:36, 17.60s/it]

Filled staff info for HK Electric Investments: WAN Chi Tin - Managing Director
Filled staff info for HK Electric Investments: WONG Tak Choy - Executive Director, Operations
Filled staff info for HK Electric Investments: WONG Lap Keung, Raymond - Executive Director, Customer Services


50it [11:56, 14.33s/it]

Filled staff info for Want Want China: TSAI Eng-Meng - Chairman & CEO
Filled staff info for Want Want China: TSAI Shao-Chung - Executive Director & Vice Chairman


In [31]:
df_staff

,Company ID,Contact Name,Role,Connection Strength,Previous HKJC Interaction,Link of HKJC Interaction evidance,Linkedin
0,101,Edward Bell,"General Manager Brand, Insights, and Marketing",Hot,"Company sponsored Longines Masters of HK, an e...",https://www.longinesmasters.com/en/news/longin...,https://www.linkedin.com/in/ecbell30
1,101,Ronald Lam,Chief Executive Officer,Hot,"As CEO, oversees major corporate partnerships....",https://www.longinesmasters.com/en/news/longin...,https://www.linkedin.com/in/ronald-lam-180b06109
2,101,Lavinia Lau,Chief Customer and Commercial Officer,Hot,Her role encompasses brand partnerships. Compa...,https://www.longinesmasters.com/en/news/longin...,https://www.linkedin.com/in/lavinia-lau-38b42217
3,102,Amy Lo,"Co-Head of Wealth and Personal Banking, Asia P...",Hot,Presented HSBC Premier Cup and gave a public s...,https://campaigns.hkjc.com/racing-news/english...,https://hk.linkedin.com/in/amy-lo-281b3127
4,102,Edward Hui,"Head of Brand and Marketing, Hong Kong",Hot,Oversees HSBC's long-standing sponsorships wit...,https://campaigns.hkjc.com/racing-news/english...,https://hk.linkedin.com/in/edwardhui-marketing
...,...,...,...,...,...,...,...
137,149,WAN Chi Tin,Managing Director,Hot,Yes - Officiated opening of Jockey Club Smart ...,https://campaign.hkjc.com/en/news/2021-09/news...,https://www.linkedin.com/in/wan-chi-tin-b40b075a/
138,149,WONG Tak Choy,"Executive Director, Operations",Warm,Nil,Nil,https://www.linkedin.com/in/wong-tak-choy-2747...
139,149,"WONG Lap Keung, Raymond","Executive Director, Customer Services",Warm,Nil,Nil,https://www.linkedin.com/in/raymond-wong-b040315/
140,150,TSAI Eng-Meng,Chairman & CEO,Cold,Nil,Nil,Nil


In [32]:
df_staff.to_csv("data_source/staff_data_filled.csv", index=False)

# Update the Database

### Create Table Schemas

In [ ]:
# # Create table schemas for the three dataframes
# ddl_statements = [
#     """
#     CREATE TABLE IF NOT EXISTS companies_csv (
#         "Rank" BIGINT,
#         "Name" TEXT,
#         "Symbol" TEXT,
#         marketcap DOUBLE PRECISION,
#         "price (HKD)" DOUBLE PRECISION,
#         country TEXT
#     );
#     """,
#     """
#     CREATE TABLE IF NOT EXISTS data_company (
#         company_name TEXT,
#         contact_person TEXT,
#         job_title TEXT,
#         website TEXT,
#         company_size BIGINT,
#         industry TEXT,
#         email TEXT,
#         phone TEXT
#     );
#     """,
#     """
#     CREATE TABLE IF NOT EXISTS staff_related (
#         "Name" TEXT,
#         "Title" TEXT,
#         linkedin_url TEXT,
#         address TEXT,
#         company_name TEXT,
#         email TEXT,
#         phone TEXT
#     );
#     """
# ]

# with engine.begin() as conn:
#     for ddl in ddl_statements:
#         conn.execute(text(ddl))

# print("Schemas created (or already existed): companies_csv, data_company, staff_related")

Schemas created (or already existed): companies_csv, data_company, staff_related


### Check the tables in DB

In [ ]:
# # List user tables in the current PostgreSQL database
# with engine.connect() as c:
#     tables = c.execute(text("""
#         SELECT table_schema, table_name
#         FROM information_schema.tables
#         WHERE table_type = 'BASE TABLE'
#           AND table_schema NOT IN ('pg_catalog', 'information_schema')
#         ORDER BY table_schema, table_name
#     """)).fetchall()

# pd.DataFrame(tables, columns=["schema", "table_name"])

,schema,table_name
0,public,companies_csv
1,public,data_company
2,public,staff_related


### Delete and Re-import the df data into the db tables

In [ ]:
# # Delete existing rows and re-import all dataframe data into PostgreSQL tables
# with engine.begin() as conn:
#     conn.execute(text("""
#         TRUNCATE TABLE companies_csv, data_company, staff_related;
#     """))

#     df_companies.to_sql("companies_csv", con=conn, if_exists="append", index=False, method="multi")
#     df_data_company.to_sql("data_company", con=conn, if_exists="append", index=False, method="multi")
#     df_staff_related.to_sql("staff_related", con=conn, if_exists="append", index=False, method="multi")

# # Quick validation
# pd.read_sql("""
#     SELECT 'companies_csv' AS table_name, COUNT(*) AS row_count FROM companies_csv
#     UNION ALL
#     SELECT 'data_company', COUNT(*) FROM data_company
#     UNION ALL
#     SELECT 'staff_related', COUNT(*) FROM staff_related
#     ORDER BY table_name
# """, con=engine)

,table_name,row_count
0,companies_csv,151
1,data_company,5
2,staff_related,25


In [ ]:
# pd.read_sql("SELECT * FROM staff_related LIMIT 5", con=engine)

,Name,Title,linkedin_url,address,company_name,email,phone
0,Melissa Wong,Chief Customer & Marketing Officer,https://www.linkedin.com/in/melissa-wong-4ba0374,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong and Macau,,
1,Alger Fung,Chief Executive Officer,https://www.linkedin.com/in/alger-fung-11531210b,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong and Macau,,
2,Stuart A. Spencer,Chief Marketing Officer,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Group,,
3,Amy Lo,Head and Chief Executive,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong,,
4,Esther Chan,Communications Manager,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong,,+852 2100 1416
